# 12 · Quantization Pipeline

Exports trained models to ONNX (FP32) and applies static INT8 QDQ quantization
for STM32 deployment via X-CUBE-AI.

**Why quantization matters for STM32 (unlike pruning):**
- INT8 reduces model size by ~4× (float32 → int8)
- STM32 Cortex-M cores with DSP extension execute INT8 MAC operations natively
- X-CUBE-AI generates optimized INT8 code from QDQ ONNX models
- Real speedup and memory reduction on target hardware

Run this notebook once per model variant you want to deploy.
Set MODEL_TAG to identify the variant in output filenames.


In [ ]:
# ── Mount Drive & load utils ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sys, shutil, os
UTILS_SRC = "/content/drive/My Drive/thesis/utils"
if os.path.exists(UTILS_SRC):
    shutil.copytree(UTILS_SRC, "/content/utils", dirs_exist_ok=True)
    sys.path.insert(0, "/content")
    print("✅ utils loaded from Drive")
else:
    sys.path.insert(0, "/content")
    print("⚠️  Place the utils/ folder at: My Drive/thesis/utils/")


In [ ]:
# ── Standard imports ────────────────────────────────────────────────
import os, time, random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from utils.dataset import prepare_dataset, get_loaders, get_test_loader
from utils.models  import (
    VGG_Scratch, VGG_Pretrained,
    ResNet_Scratch, ResNet18_Pretrained,
    MobileNetV2_Scratch, MobileNetV3_Pretrained,
    count_params, model_size_mb, MODEL_REGISTRY,
)
from utils.train import (
    setup_device, set_seed, evaluate,
    train_model, train_model_three_phase,
    train_multi_seed, train_kd, plot_history,
)

device = setup_device(seed=41)


In [ ]:
# ── Dataset setup ───────────────────────────────────────────────────
prepare_dataset()


In [ ]:
!pip -q install onnx onnxruntime onnxscript
import onnx
import onnxruntime as ort
from onnxruntime.quantization import quantize_static, QuantType, QuantFormat, CalibrationDataReader
import shutil, numpy as np
from pathlib import Path
from utils.dataset import manifest_paths, VWWDataset, get_eval_transform
from torch.utils.data import DataLoader


In [ ]:
SAVE_DIR = "/content/drive/My Drive/Colab Notebooks"


In [ ]:
# ── Configuration ─────────────────────────────────────────────────────
# Change MODEL_TAG and CKPT_PATH for each variant you export

MODEL_TAG  = "kd_ft"                                    # identifier for output files
CKPT_PATH  = f"{SAVE_DIR}/kd_ft_seed_XX.pth"           # checkpoint to export
MODEL_FN   = MobileNetV3_Pretrained                     # must match checkpoint

EXPORT_DIR    = Path(f"{SAVE_DIR}/exports_{MODEL_TAG}"); EXPORT_DIR.mkdir(exist_ok=True)
EVAL_SAMPLES  = 200
CALIB_SAMPLES = 200
IMG_SIZE      = 96

FP32_ONNX = f"vww_{MODEL_TAG}_fp32.onnx"
INT8_ONNX = f"vww_{MODEL_TAG}_int8_qdq.onnx"
CALIB_NPZ = f"vww_{MODEL_TAG}_calib_{CALIB_SAMPLES}.npz"
VAL_NPZ   = f"vww_{MODEL_TAG}_val_{EVAL_SAMPLES}.npz"
LABEL_NPZ = f"vww_{MODEL_TAG}_labels_{EVAL_SAMPLES}.npz"


In [ ]:
# ── Load and verify model ─────────────────────────────────────────────
model = MODEL_FN().to(device)
model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
model.eval()

_, val_loader = get_loaders(batch_size=64)
acc = evaluate(model, val_loader, device)
print(f"PyTorch val accuracy: {acc*100:.2f}%  (checkpoint: {CKPT_PATH})")


In [ ]:
# ── Export FP32 ONNX ──────────────────────────────────────────────────
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
torch.onnx.export(
    model, dummy, FP32_ONNX,
    input_names=["input"], output_names=["logits"],
    opset_version=18, do_constant_folding=True,
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
    dynamo=False,
)
onnx.checker.check_model(FP32_ONNX)
print("✅ FP32 ONNX:", FP32_ONNX)


In [ ]:
# ── Collect calibration and validation arrays ─────────────────────────
mp      = manifest_paths()
eval_tf = get_eval_transform()

val_ds   = VWWDataset(mp["val_person"],   mp["val_non_person"],   eval_tf)
train_ds = VWWDataset(mp["train_person"], mp["train_non_person"], eval_tf)

def collect(ds, n):
    loader = DataLoader(ds, batch_size=1, shuffle=False)
    xs, ys = [], []
    for i, (x, y) in enumerate(loader):
        if i >= n: break
        xs.append(x.numpy().astype("float32")[0])
        ys.append(int(y.item()))
    return np.stack(xs), np.array(ys, dtype="int32")

val_x,   val_y   = collect(val_ds,   EVAL_SAMPLES)
calib_x, _       = collect(train_ds, CALIB_SAMPLES)

np.savez(VAL_NPZ,   input=val_x)
np.savez(LABEL_NPZ, label=val_y)
np.savez(CALIB_NPZ, input=calib_x)
print(f"Val:   {val_x.shape}  |  Calib: {calib_x.shape}")


In [ ]:
# ── Static INT8 QDQ quantization ──────────────────────────────────────
class CalibReader(CalibrationDataReader):
    def __init__(self, npz, name="input"):
        self.x = np.load(npz)["input"]; self.name = name; self.i = 0
    def get_next(self):
        if self.i >= len(self.x): return None
        out = {self.name: self.x[self.i:self.i+1]}; self.i += 1; return out
    def rewind(self): self.i = 0

quantize_static(
    model_input=FP32_ONNX, model_output=INT8_ONNX,
    calibration_data_reader=CalibReader(CALIB_NPZ),
    quant_format=QuantFormat.QDQ,
    activation_type=QuantType.QInt8, weight_type=QuantType.QInt8,
    per_channel=True,
)
print("✅ INT8 QDQ ONNX:", INT8_ONNX)


In [ ]:
# ── ONNX Runtime accuracy verification ───────────────────────────────
def onnx_accuracy(onnx_path, val_x, val_y):
    sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
    in_name  = sess.get_inputs()[0].name
    out_name = sess.get_outputs()[0].name
    preds = [np.argmax(sess.run([out_name], {in_name: val_x[i:i+1]})[0][0])
             for i in range(len(val_x))]
    return (np.array(preds) == val_y).mean() * 100

fp32_ort_acc = onnx_accuracy(FP32_ONNX, val_x, val_y)
int8_ort_acc = onnx_accuracy(INT8_ONNX, val_x, val_y)
print(f"ONNX FP32 accuracy: {fp32_ort_acc:.2f}%")
print(f"ONNX INT8 accuracy: {int8_ort_acc:.2f}%")
print(f"INT8 accuracy drop: {fp32_ort_acc - int8_ort_acc:.2f}%")


In [ ]:
# ── Copy all exports to Drive ─────────────────────────────────────────
for f in [FP32_ONNX, INT8_ONNX, CALIB_NPZ, VAL_NPZ, LABEL_NPZ]:
    if os.path.exists(f):
        shutil.copy2(f, EXPORT_DIR / f)
        print(f"✅ {f} → {EXPORT_DIR}")

print(f"\nExport folder: {EXPORT_DIR}")


## STM32 deployment instructions

1. Open X-CUBE-AI in STM32CubeIDE
2. Import `vww_*_fp32.onnx` for FP32 inference
3. Import `vww_*_int8_qdq.onnx` for INT8 inference (recommended)
4. Use `vww_*_val_200.npz` as input batch for both validation runs
5. After each run, rename `network_val_io.npz`:
   - FP32 run → `stm32_outputs_fp32.npz`
   - INT8 run → `stm32_outputs_int8.npz`
6. Run `compute_stm32_accuracy()` below to score the results


In [ ]:
def compute_stm32_accuracy(labels_npz, outputs_npz, key="c_outputs_1", num_classes=2):
    """Score STM32 output NPZ against ground-truth labels."""
    y      = np.load(labels_npz)["label"].astype("int64")
    raw    = np.load(outputs_npz)[key].reshape(len(y), num_classes)
    acc    = (np.argmax(raw, 1) == y).mean() * 100
    print(f"STM32 accuracy: {acc:.2f}%  ({len(y)} samples)")
    return acc

# Uncomment after STM32 runs:
# compute_stm32_accuracy(LABEL_NPZ, "stm32_outputs_fp32.npz")
# compute_stm32_accuracy(LABEL_NPZ, "stm32_outputs_int8.npz")
